## Bienvenue jour 02

In [7]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API Key was found - please head over to the troubleshooting notebook in this folder to identify & fix")
elif not api_key.startswith("sk-proj-"):
    print("An API Key was found, but it doesn't start with sk-proj-")
else:   
    print("API Key found and looks good so far!")

API Key found and looks good so far!


## Decoutvrons les Endpoints

les open de OpenAI sont les premier a etre utiliser et beaucoup d'autres ont simplement suivie

In [2]:
import requests

openai_model = os.getenv("OPENAI_MODEL")

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": openai_model,
    "messages": [
        {
            "role": "user",
            "content": "Hello, how are you doing today?"
        }
    ]
}

payload

{'model': 'gpt-4o-mini',
 'messages': [{'role': 'user', 'content': 'Hello, how are you doing today?'}]}

In [9]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions", 
    headers=headers, 
    json=payload
)

In [10]:
response.json()["choices"][0]["message"]["content"]

KeyError: 'choices'

## Le paquet de OpenAI

In [ ]:
openai = OpenAI()

response = openai.chat.completions.create(
    model=openai_model,
    messages=[{
        "role": "user",
        "content": "What are the business applications of generative AI?"
    }]
)

response.choices[0].message.content

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

## Et puis, cette chose géniale est arrivée :

L'API Chat Completions d'OpenAI a été si populaire que les autres fournisseurs de modèles ont créé des points de terminaison (endpoints) identiques.

Ils sont connus sous le nom d'« OpenAI Compatible Endpoints » (points de terminaison compatibles OpenAI).

Par exemple, Google en a créé un ici : https://generativelanguage.googleapis.com/v1beta/openai/

Et OpenAI a décidé d'être sympa : ils ont dit, hé, vous pouvez simplement utiliser la même bibliothèque client que celle que nous avons faite pour GPT. Nous vous permettrons de spécifier une URL de point de terminaison différente et une clé différente, pour utiliser un autre fournisseur.

Vous pouvez donc utiliser :

gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)

Et pour être clair — même si OpenAI est présent dans le code, nous utilisons uniquement cette bibliothèque client Python légère pour appeler le point de terminaison — aucun modèle OpenAI n'est impliqué ici.

Si vous êtes confus, veuillez consulter le Guide 9 dans le dossier Guides !

Et maintenant, essayons !

In [8]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai"

# Récupération de la clé
google_api_key = os.getenv("GEMINI_API_KEY")

if not google_api_key:
    print("Aucune clé API trouvée - Vérifie ton fichier .env")
# Correction ici : .startswith() au lieu de .startswitch()
elif not google_api_key.startswith("AIza"): 
    print("Clé trouvée, mais elle ne semble pas valide (doit commencer par AIza)")
else:
    print("Clé API trouvée et format valide !")

Clé API trouvée et format valide !


In [ ]:
MODEL = os.getenv("GEMINI_API_KEY")

In [10]:
genai = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = genai.chat.completions.create(
    model=MODEL, 
    messages=[{"role": "user", "content": "What are the business applications of generative AI?"}]
)

response.choices[0].message.content

'Generative AI (GenAI) is transforming the business landscape by moving beyond simple automation to **augmentation**—enhancing human creativity and decision-making. \n\nHere are the primary business applications of Generative AI, categorized by functional area:\n\n### 1. Marketing and Content Creation\nThis is currently the most widespread application of GenAI.\n*   **Copywriting at Scale:** Generating blog posts, social media captions, email newsletters, and ad copy tailored to specific brand voices.\n*   **Visual Asset Generation:** Using tools like Midjourney or DALL-E to create high-quality stock photography, marketing illustrations, and social media graphics instantly.\n*   **Personalization:** Creating thousands of variations of a single campaign to target specific customer personas based on their interests and past behavior.\n*   **SEO Optimization:** Generating meta-descriptions, keyword-rich headers, and structured data to improve search rankings.\n\n### 2. Sales and Customer 

## LES ENDPOINT DE OPENAI EST AUSSI COMPATINLE AVEC OLLAMA

In [12]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Telecharger llama3.2:1b (le modele de meta) si ce n'est pas deja fait 

In [22]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama") 

In [23]:
system_prompt = " Tu es un expert en maketing, IA et un entrepreneur brillant \
qui répond de manière concise et claire et donne des conseil precise et realist \
en utilisant le markdown pour formater les réponses."

In [24]:
def user_prompt_for(topic):
    user_prompt = f"Dans le domaine suivant: {topic}"
    user_prompt += "\nQuels sont les meilleurs de Business \
en utilisant des meilleurs pratiques les meme que les millionaire \
ont utilise pour en arrive la ?"
    return user_prompt

In [26]:
print("MESSAGES ENVOYER A LLM:")
topic = "l'IA générative dans le business"
print(f"{user_prompt_for(topic)}")

MESSAGES ENVOYER A LLM:
Dans le domaine suivant: l'IA générative dans le business
Quels sont les meilleurs de Business en utilisant des meilleurs pratiques les meme que les millionaire ont utilise pour en arrive la ?


In [30]:
def messages_for(topic):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(topic)}
    ]

In [31]:
def get_response(topic):
    messages = messages_for(topic)
    response = ollama.chat.completions.create(
        model="llama3.2:1b", 
        messages=messages
    )
    return response.choices[0].message.content

In [32]:
get_response(topic)

"Voici quelques-uns des meilleurs exemples de startups et d'entreprises qui ont réussi à utiliser l'IA générative dans leur business :\n\n1. **Netflix** : Achat d'une collection complète de films sans avoir besoin de regarder les derniers épisodes pour terminer la série télévisée.\n\n    * Practique utilise :\n        1. Analyse du contenu : Analyser la concurrence, les modèles en cours et les tendances du marché afin d'augmenter la vitesse des créations.\n        2. Pré-processing de données : Utiliser l'intelligence artificielle pour prétraiter les données d'images pour augmenter la qualité des fichiers.\n    2. **Dropbox** : Stockage cloud basé sur les données numériques à portée de main, permettant aux clients de conserver leurs documents et réseaux sans avoir besoin de copier sur leur ordinateur ou disque dur.\n\n    * Practique utilise :\n        1. Utilisation de couches de données : Créer des modèles de données afin d'analyser les transactions.\n        2. Gestion du contenu : 

In [33]:
#Afficher la reponse proprement
from IPython.display import Markdown, display
def display_summary(topic):
    resultat = get_response(topic)
    display(Markdown(resultat))

In [ ]:
display_summary(topic)

Voici quelques-uns des meilleurs exemples d'entrepreneurs réussis qui ont utilisé des pratiques de l'IA générative dans leur businesses, ainsi qu'une analyse de leurs principaux points :

**Exemples :**

1. **Google** :
 * Le dirigeant en chef, Sundar Pichai, utilisait des méthodes d'apprentissage automatique ( Machine learning et deep learning ) pour améliorer l'expérience utilisateur et la recherche.
2. **Uber** :
 * Le CEO, Travis Kalanick, a créé une équipe multidisciplinaire composée de données scientists, développeurs techniques et designers pour mettre en œuvre des modèles d'apprentissage automatique.
3. **Airbnb** :
 * La PDG, Brian Chesky, utilise des algorithmes intelligents pour déterminer les propositions d'hôtel proposées aux clients.
4. **Amazon** :
 * Le CEO, Jeff Bezos, a créé un centre de recherche sur l'IA et l'AUT (Artificielle Intelligence) où les data scientists peuvent travailler ensemble avec les développeurs techniques pour mettre en œuvre les dernières innovations.

**Pratiques employées :**

1. **Formation et suivi**
2. **Développement d'expérience utilisateur** (UX)
3. **Optimisation des processus**
4. **Amélioration continue de la connaissance** (Machine learning, AI)
5. **Collaboration entre équipes multidisciplinaires**

**Analyse:**

Ces examples montrent comment les entrepreneurs ont pu révolutionner leurs entreprises en utilisant des pratiques de l'IA générative.

* Elles mettent en avant la nécessité de créer une équipe multidisciplinaire composée de spécialistes pour concevoir et mettre en œuvre des systèmes d'apprentissage automatique.
* Le rôle important d'une expertise approfondie en données sciences, délivrance techniques et conception utilisateur.
* L'importance des algorithmes intelligents dans l'amélioration continue de la connaissance et de la conformation aux besoins changants du marché.

**Conseils:**

1. Raissez les compétences requises pour travailler avec l'IA générative (machine learning, AI, données sciences).
2. Creaz un équipe multidisciplinaire pour concevoir et mettre en œuvre des solutions.
3. Améliorez continuellement vos connaissances pour rester à jour de la dernière innovation technologique.

Ces conseils peuvent aider les entrepreneurs réussis à adopter les meilleures pratiques pour améliorer leur entreprise en utilisant des méthodes de l'IA générative.